In [4]:
from datamodules.bert_datamodule import BertDataModule
import hydra
from models.bert_module import CustomBertModelModule
import pytorch_lightning as pl
from pytorch_lightning import loggers as pl_loggers
from aggregator import Aggregator
from client import Client
import torch.multiprocessing as mp
import torch
import copy
import uuid

In [5]:
# Hyperparameters
num_clients = 2
num_rounds = 2
num_epochs = 5

# Module instantiation
datamodule = BertDataModule(glue_dataset="sst2",
                            batch_size=32,
                            num_workers=1,
                            model_type="bert-base-uncased",
                            truncate=100,
                            num_splits=num_clients)

# Data split
datamodule.prepare_data()
datamodule.setup()
train_dls = datamodule.train_dataloader()
val_dls = datamodule.val_dataloader()

model = CustomBertModelModule.from_pretrained("bert-base-uncased", num_labels=2)

# Instantiate clients 
clients = []
for i in range(num_clients):
    client = Client(id=f"client_{i+1}",
                    model=copy.deepcopy(model),
                    trainer=pl.Trainer(default_root_dir="lightning_logs/",
                                       min_epochs=1,
                                       max_epochs=num_epochs,
                                       accelerator="gpu",
                                       devices=1,
                                       limit_train_batches=100,
                                       limit_val_batches=10,
                                       limit_test_batches=10,
                                       check_val_every_n_epoch=1,
                                       deterministic=False),
                    train_data=train_dls[i],
                    val_data=val_dls[i])
    clients.append(client)

# Instantiate aggregator
aggregator = Aggregator(name="sfl_aggregator")


# Compare models
def models_equal(model1, model2):
    for (name1, param1), (name2, param2) in zip(model1.named_parameters(), model2.named_parameters()):
        if name1.startswith("bert.encoder") or name1.startswith("classifier") or name1.startswith("bert.pooler") or name1.startswith("bert.embeddings"):
            if torch.any(param1 != param2):
                print("NOT EQUAL")
                return False
    return True

def print_diffs(model1, model2):
    for (name1, param1), (name2, param2) in zip(model1.named_parameters(), model2.named_parameters()):
        if name1.startswith("bert.encoder") or name1.startswith("classifier") or name1.startswith("bert.pooler") or name1.startswith("bert.embeddings"):
            if torch.any(param1 != param2):
                print(name1)


Found cached dataset glue (/home/aladin/.cache/huggingface/datasets/glue/sst2/1.0.0/dacbe3125aa31d7f70367a07a8a9e72a5a0bfeb5fc42e75c9db75b96da6053ad)
100%|██████████| 3/3 [00:00<00:00, 850.94it/s]
Some weights of the model checkpoint at bert-base-uncased were not used when initializing CustomBertModelModule: ['cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.decoder.weight']
- This IS expected if you are initializing CustomBertModelModule from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing CustomBertModelModule from the checkpoint of a model that you expect to be exactly identical (initializing a Ber

In [6]:
# Before Training
model1 = clients[0].model
model2 = clients[1].model

if models_equal(model1, model2):
    print("Models are equal before training")
else:
    print("Models are not equal before training")
    print_diffs(model1, model2)

Models are equal before training


In [7]:
for client in clients:
    client.trainer = pl.Trainer(default_root_dir="lightning_logs/",
                                        min_epochs=1,
                                        max_epochs=num_epochs,
                                        accelerator="gpu",
                                        devices=1,
                                        limit_train_batches=100,
                                        limit_val_batches=10,
                                        limit_test_batches=10,
                                        check_val_every_n_epoch=1,
                                        deterministic=False)
    client.trainer.fit(client.model, client.train_data, client.val_data)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]

  | Name       | Type              | Params
-------------------------------------------------
0 | bert       | BertModel         | 109 M 
1 | dropout    | Dropout           | 0     
2 | classifier | Linear            | 1.5 K 
3 | embedding  | BertModel         | 109 M 
4 | attention  | CustomBertEncoder | 85.1 M
5 | accuracy   | BinaryAccuracy    | 0     
-------------------------------------------------
304 M     Trainable params
0         Non-trainable params
304 M     Total params
1,216.082 Total estimated model params size (MB)


Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]

/home/aladin/projects/resilient_sfl/venv/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:224: PossibleUserWarning: The dataloader, val_dataloader 0, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 96 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.
  rank_zero_warn(


/home/aladin/projects/resilient_sfl/venv/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:224: PossibleUserWarning: The dataloader, train_dataloader, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 96 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.
  rank_zero_warn(
/home/aladin/projects/resilient_sfl/venv/lib/python3.10/site-packages/pytorch_lightning/trainer/trainer.py:1558: PossibleUserWarning: The number of training batches (2) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.
  rank_zero_warn(


Epoch 4: 100%|██████████| 4/4 [00:00<00:00, 11.33it/s, loss=0.944, v_num=12, val_loss=0.797, val_acc=0.420, train_loss=0.805, train_acc=0.500]

...

Epoch 4: 100%|██████████| 4/4 [00:02<00:00,  1.44it/s, loss=0.944, v_num=12, val_loss=0.797, val_acc=0.420, train_loss=0.805, train_acc=0.500]


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]

  | Name       | Type              | Params
-------------------------------------------------
0 | bert       | BertModel         | 109 M 
1 | dropout    | Dropout           | 0     
2 | classifier | Linear            | 1.5 K 
3 | embedding  | BertModel         | 109 M 
4 | attention  | CustomBertEncoder | 85.1 M
5 | accuracy   | BinaryAccuracy    | 0     
-------------------------------------------------
304 M     Trainable params
0         Non-trainable params
304 M     Total params
1,216.082 Total estimated model params size (MB)


Epoch 4: 100%|██████████| 4/4 [00:00<00:00, 10.85it/s, loss=0.837, v_num=13, val_loss=0.805, val_acc=0.540, train_loss=0.794, train_acc=0.520]

`Trainer.fit` stopped: `max_epochs=5` reached.


Epoch 4: 100%|██████████| 4/4 [00:02<00:00,  1.44it/s, loss=0.837, v_num=13, val_loss=0.805, val_acc=0.540, train_loss=0.794, train_acc=0.520]


In [8]:
# After Training
model1 = clients[0].model
model2 = clients[1].model

if models_equal(model1, model2):
    print("Models are equal after training")
else:
    print("Models are not equal after training")
    print_diffs(model1, model2)

NOT EQUAL
Models are not equal after training
classifier.weight
classifier.bias


In [ ]:
if models_equal(client_ref.model, client.trainer.model):
    print("The models are equal!")
else:
    print("The models are not equal!")

In [ ]:
print(type(client.trainer.model))

In [ ]:
if models_equal(client.trainer.model, clients[0].model):
    print("The models are equal!")
else:
    print("The models are not equal!")

In [ ]:
new_model = CustomBertModelModule.from_pretrained("bert-base-uncased", num_labels=2)

if models_equal(client.trainer.model, new_model):
    print("The models are equal!")
else:
    print("The models are not equal!")

In [ ]:
# SFL global round loop
for r in range(num_rounds):
    
    print(f"GLOBAL ROUND : {r+1} of {num_rounds}")

    # Train client models
    for client in clients:
        # Reset trainer to avoid max_epochs boundary
        client.trainer = pl.Trainer(default_root_dir="lightning_logs/",
                                       min_epochs=1,
                                       max_epochs=num_epochs,
                                       accelerator="gpu",
                                       devices=1,
                                       limit_train_batches=100,
                                       limit_val_batches=10,
                                       limit_test_batches=10,
                                       check_val_every_n_epoch=1,
                                       deterministic=False)
        client.trainer.fit(client.model, client.train_data, client.val_data)
        client.model = client.trainer.model

    print("ALL CLIENTS TRAINED")
    if models_equal(clients[0].model, clients[1].model):
        print("The models are equal!")
    else:
        print("The models are not equal!")

    # Aggregate client models
    attentions = aggregator.accumulate_attentions([client.model for client in clients])
    heads = aggregator.accumulate_heads([client.model for client in clients])
    embeddings = aggregator.accumulate_embeddings([client.model for client in clients])

    aggregated_attentions = aggregator.aggregate(attentions)
    aggregated_heads = aggregator.aggregate(heads)
    aggregated_embeddings = aggregator.aggregate(embeddings)

    # Model update and save
    for client in clients:
        client.update_model(aggregated_attentions)
        client.update_model(aggregated_heads)
        client.update_model(aggregated_embeddings)

    print("ALL CLIENTS UPDATED")
    if models_equal(clients[0].model, clients[1].model):
        print("The models are equal!")
    else:
        print("The models are not equal!")

In [ ]:
test_model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

if models_equal(test_model, clients[0].model):
    print("The models are equal!")
else:
    print("The models are not equal!")

In [ ]:
# Aggregate client models
attentions = aggregator.accumulate_attentions([client.model for client in clients])
heads = aggregator.accumulate_heads([client.model for client in clients])
embeddings = aggregator.accumulate_embeddings([client.model for client in clients])

aggregated_attentions = aggregator.aggregate(attentions)
aggregated_heads = aggregator.aggregate(heads)
aggregated_embeddings = aggregator.aggregate(embeddings)

In [ ]:
import torch
a = attentions[0]
b = attentions[1]

for key in a.keys():
    if not torch.allclose(a[key], b[key]):
        print(key)
    else:
        pass


# print(a['bert.encoder.layer.0.attention.self.query.weight'] == b['bert.encoder.layer.0.attention.self.query.weight'])